# Dev notebook — nowcast() / iaqi_hour() / iaqi_day()

Viết và test trước khi port vào `iaqi.py`. Mọi số kỳ vọng lấy trực tiếp từ QĐ 1459/QĐ-TCMT (mục 2.3 — Ví dụ tính mẫu).

**Lưu ý:** notebook này để tham khảo lại lịch sử làm việc (đã port xong vào `iaqi.py`/`test_iaqi.py` thật). Nếu muốn tránh lệch code, nên đổi các cell hàm bên dưới thành `from aqi_core.iaqi import nowcast, iaqi_hour, iaqi_day, level_of` thay vì định nghĩa lại.

In [ ]:
import json
from typing import Optional

with open("breakpoints_vn.json", encoding="utf-8") as f:
    STD = json.load(f)

BP = STD["pollutants"]
CALC_BANDS = STD["aqi_breakpoints"]   # 7 khoảng tính toán (i_low/i_high) — Bảng 2
LEVEL_LABELS = STD["level_labels"]    # 6 mức hiển thị — Bảng 1

print("pollutants:", list(BP.keys()))
print("calc bands:", len(CALC_BANDS), "| level labels:", len(LEVEL_LABELS))

## Core: `iaqi()` + `level_of()`

Giữ nguyên logic nội suy tuyến tính của `iaqi.py` hiện tại — chỉ đổi nguồn tra `level`/`label` sang `level_labels` (vì `aqi_breakpoints` giờ chỉ còn `i_low`/`i_high`).

In [ ]:
def iaqi(concentration: Optional[float], pollutant: str) -> Optional[float]:
    if concentration is None or concentration < 0:
        return None
    bp = BP[pollutant]["bp"]
    for i in range(len(bp) - 1):
        if bp[i] <= concentration <= bp[i + 1]:
            bp_low, bp_high = bp[i], bp[i + 1]
            i_low, i_high = CALC_BANDS[i]["i_low"], CALC_BANDS[i]["i_high"]
            return (i_high - i_low) / (bp_high - bp_low) * (concentration - bp_low) + i_low
    return float(CALC_BANDS[-1]["i_high"])


def level_of(aqi_value: float) -> dict:
    for lv in LEVEL_LABELS:
        if aqi_value <= lv["i_high"]:
            return {"level": lv["level"], "label": lv["label"]}
    return {"level": 6, "label": "Nguy hại"}

## `nowcast()` — mục 2.2.1a

`hourly_values[0]` = giờ hiện tại (c1), `hourly_values[-1]` = xa nhất (tối đa c12).

Rule: cần ít nhất 2/3 giá trị đầu (c1,c2,c3) có dữ liệu, không thì trả `None`.

In [ ]:
def nowcast(hourly_values: list[Optional[float]]) -> Optional[float]:
    if sum(v is not None for v in hourly_values[:3]) < 2:
        return None
    present = [(i, v) for i, v in enumerate(hourly_values) if v is not None]
    if not present:
        return None
    vals = [v for _, v in present]
    w_star = min(vals) / max(vals)
    w = w_star if w_star > 0.5 else 0.5
    num = sum((w ** i) * v for i, v in present)
    den = sum((w ** i) for i, _ in present)
    return num / den

### Test Nowcast — đối chiếu mục 2.3a

Bảng gốc liệt kê PM2.5 từ 09:00 → 20:00. c1 (hiện tại) = 20:00, nên phải **đảo ngược** trước khi truyền vào `nowcast()`.

In [ ]:
pm25_09h_to_20h = [26.9, 24.7, 20.5, 23.5, 19.5, 16.5, 19.0, 16.5, 20.3, 22.4, 19.6, 20.6]
c1_to_c12 = list(reversed(pm25_09h_to_20h))

nc = nowcast(c1_to_c12)
print("Nowcast =", round(nc, 1), "(kỳ vọng 20.3)")
assert abs(nc - 20.3) < 0.1

## `iaqi_hour()` — Công thức 1/2, mục 2.2.1b

`components`: `pm2_5`/`pm10` đã là Nowcast (tính ở bước trên); `so2`/`no2`/`co`/`o3` đã là TB1h hiện tại.

In [ ]:
POLLUTANT_COLS_HOUR = ["pm2_5", "pm10", "so2", "no2", "co"]


def _reduce(subs: dict) -> dict:
    valid = {p: v for p, v in subs.items() if v is not None}
    if not valid:
        return {"aqi": None, "aqi_level": None, "aqi_label": None,
                "dominant_pollutant": None, "iaqi": subs}
    dom = max(valid, key=valid.get)
    value = valid[dom]
    lvl = level_of(value)
    return {"aqi": round(value), "aqi_level": lvl["level"], "aqi_label": lvl["label"],
            "dominant_pollutant": dom, "iaqi": subs}


def iaqi_hour(components: dict) -> dict:
    subs = {p: iaqi(components.get(p), p) for p in POLLUTANT_COLS_HOUR}
    subs["o3"] = iaqi(components.get("o3"), "o3_1h")
    return _reduce(subs)

### Test AQI giờ — đối chiếu mục 2.3b

Kỳ vọng theo văn bản: AQI(O3)=43, AQI(NO2)=60, AQI(PM2.5, Nowcast=20.3)=41 → AQI^h tổng hợp = 60.

**Đã xác nhận (kiểm tra bản gốc):** cả `C=118,7` lẫn kết quả `= 60` đều in đúng như vậy trong QĐ 1459 (mục 2.3b) — không phải lỗi OCR/transcription.

Nhưng đúng Công thức 1: `(100-50)/(200-100)*(118.7-100)+50 = 59.35` → làm tròn chuẩn = **59**, không phải 60.

Đối chiếu chéo với ví dụ AQI ngày (2.3c, NO2=130.8 → 65.4 → văn bản ghi đúng 65) loại trừ khả năng VN_AQI dùng luật làm tròn lên (ceiling) — nếu có, ví dụ ngày phải ra 66. Kết luận: ví dụ giờ (2.3b) là **lỗi số học trong chính văn bản gốc QĐ 1459**, không phải lỗi công thức/breakpoint ở đây. Test dưới đây theo ĐÚNG công thức (59), không ép theo số 60 sai.

In [ ]:
out_hour = iaqi_hour({"o3": 136.1, "no2": 118.7, "pm2_5": 20.3})
print(out_hour)

assert round(out_hour["iaqi"]["o3"]) == 43
assert round(out_hour["iaqi"]["no2"]) == 59
assert round(out_hour["iaqi"]["pm2_5"]) == 41
assert out_hour["aqi"] == 59

## `iaqi_day()` — mục 2.2.2, xử lý rule O3 8h > 400

`components`: `pm2_5`/`pm10` là TB24h; `so2`/`no2`/`co` là max TB1h trong ngày; `o3_1h_max`/`o3_8h_max` là max TB1h / max TB8h trong ngày.

Rule (mục 2.2.2b, ghi chú): nếu TB8h lớn nhất trong ngày > 400 µg/m³ thì **không tính** AQI O3 theo 8h — chỉ dùng O3(1h).

In [ ]:
def _iaqi_o3_day(o3_1h_max: Optional[float], o3_8h_max: Optional[float]) -> Optional[float]:
    v_1h = iaqi(o3_1h_max, "o3_1h")
    if o3_8h_max is not None and o3_8h_max <= 400:
        v_8h = iaqi(o3_8h_max, "o3_8h")
        candidates = [v for v in (v_1h, v_8h) if v is not None]
        return max(candidates) if candidates else None
    return v_1h  # >400 hoặc thiếu dữ liệu 8h -> chỉ dùng 1h


POLLUTANT_COLS_DAY = ["pm2_5", "pm10", "so2", "no2", "co"]


def iaqi_day(components: dict) -> dict:
    subs = {p: iaqi(components.get(p), p) for p in POLLUTANT_COLS_DAY}
    subs["o3"] = _iaqi_o3_day(components.get("o3_1h_max"), components.get("o3_8h_max"))
    return _reduce(subs)

### Test AQI ngày — đối chiếu mục 2.3c

Kỳ vọng: AQI(O3 8h=89.3)=45, AQI(O3 1h=114.6)=36 → O3 lấy max=45; AQI(NO2)=65; AQI(PM2.5, TB24h=55.7)=110 → AQI^d tổng hợp = **110**.

In [ ]:
out_day = iaqi_day({
    "no2": 130.8,
    "pm2_5": 55.7,
    "o3_1h_max": 114.6,
    "o3_8h_max": 89.3,
})
print(out_day)

assert round(out_day["iaqi"]["no2"]) == 65
assert round(out_day["iaqi"]["pm2_5"]) == 110
assert round(out_day["iaqi"]["o3"]) == 45
assert out_day["aqi"] == 110

## Test rule O3 8h > 400 (không có ví dụ số trong văn bản — tự dựng case biên)

In [ ]:
# TB8h = 450 > 400 -> phải bỏ qua nhánh 8h, chỉ dùng 1h
v = _iaqi_o3_day(o3_1h_max=114.6, o3_8h_max=450)
assert round(v) == round(iaqi(114.6, "o3_1h"))
print("OK: TB8h>400 bi bo qua, dung O3(1h) =", v)

# Thieu du lieu 8h -> cung phai fallback ve 1h
v2 = _iaqi_o3_day(o3_1h_max=114.6, o3_8h_max=None)
assert v2 == v
print("OK: thieu du lieu 8h, fallback O3(1h) =", v2)

## Bước tiếp theo

**Đã hoàn thành:** các hàm ở trên đã được port vào `aqi_core/iaqi.py` thật (`nowcast()`, `iaqi_hour()`, `iaqi_day()`, `_iaqi_o3_day()`, `_reduce()`), và `spark/tests/test_iaqi.py` đã cập nhật theo đúng các test ở đây (đã chạy `pytest` pass 11/11).

Notebook này giữ lại làm tài liệu tham khảo quá trình verify + phát hiện lỗi số học trong văn bản gốc QĐ 1459 (mục 2.3b, ví dụ NO2). Nếu chạy lại để test tương tác, nên đổi sang `from aqi_core.iaqi import nowcast, iaqi_hour, iaqi_day, level_of, iaqi` thay vì định nghĩa lại như trên, để không lệch với code production.